nas칼럼 채우기

In [1]:
import pandas as pd
from tqdm import tqdm

def match_ocr_to_nas(integrated_path, nas_db_path, output_path):
    print("데이터를 불러오는 중...")
    
    # 1. 엑셀 파일과 NAS CSV 데이터베이스 로드
    df_integrated = pd.read_excel(integrated_path)
    df_nas = pd.read_csv(nas_db_path)
    
    # 결측치(NaN) 방지 (문자열 처리를 위해)
    df_integrated['OCR결과'] = df_integrated['OCR결과'].fillna('')
    df_nas['파일명'] = df_nas['파일명'].fillna('')
    
    # 매칭 결과를 담을 리스트
    matched_results = []
    
    print("NAS 파일 매칭을 시작합니다...")
    # 2. integrated.xlsx의 각 'OCR결과' 값에 대해 검색 수행
    for ocr_text in tqdm(df_integrated['OCR결과'], desc="매칭 진행률"):
        ocr_text = str(ocr_text).strip()
        
        if not ocr_text:  # 빈 칸인 경우 건너뜀
            matched_results.append("")
            continue
            
        # NAS 파일명에 OCR결과 텍스트가 포함(contains)되어 있는지 확인 (대소문자 구분 없음)
        # regex=False로 두어 특수문자가 정규식으로 인식되는 에러를 방지합니다.
        matches = df_nas[df_nas['파일명'].str.contains(ocr_text, case=False, regex=False, na=False)]
        
        if not matches.empty:
            # 매칭된 NAS 파일이 여러 개일 수 있으므로 전체경로들을 줄바꿈(\n)으로 묶어서 저장
            matched_paths = "\n".join(matches['전체경로'].tolist())
            matched_results.append(matched_paths)
        else:
            matched_results.append("")  # 일치하는 항목이 없을 경우 빈 칸 유지
            
    # 3. 매칭된 결과 리스트를 원본 데이터프레임의 'NAS 매칭' 칼럼에 덮어쓰기
    df_integrated['NAS 매칭'] = matched_results
    
    # 4. 결과를 새로운 엑셀 파일로 저장
    print(f"\n매칭 완료! 결과를 '{output_path}'에 저장하는 중...")
    df_integrated.to_excel(output_path, index=False)
    print("모든 작업이 성공적으로 끝났습니다!")

if __name__ == "__main__":
    # 파일 경로 설정 (실제 환경에 맞게 경로를 수정해 주세요)
    INTEGRATED_FILE = r'output\integrated.xlsx'           # 원본 엑셀 파일
    NAS_DB_FILE = 'NAS_File_DB.csv'                       # 1단계에서 만든 NAS DB 파일
    OUTPUT_FILE = r'output\nas_matched.xlsx'       # 결과를 저장할 새로운 엑셀 파일 경로
    
    match_ocr_to_nas(INTEGRATED_FILE, NAS_DB_FILE, OUTPUT_FILE)

데이터를 불러오는 중...
NAS 파일 매칭을 시작합니다...


매칭 진행률: 100%|██████████| 918/918 [00:34<00:00, 26.35it/s]



매칭 완료! 결과를 'output\nas_matched.xlsx'에 저장하는 중...
모든 작업이 성공적으로 끝났습니다!


nas매칭 경로 제외하고 파일명만

In [ ]:
import pandas as pd
from tqdm import tqdm

def match_ocr_to_nas(integrated_path, nas_db_path, output_path):
    print("데이터를 불러오는 중...")
    
    # 1. 원본 파일과 NAS CSV 데이터베이스 로드
    # (만약 첨부해주신 것처럼 integrated.xlsx가 csv 형식으로 저장되어 있다면 pd.read_csv를 사용해야 할 수도 있습니다)
    try:
        df_integrated = pd.read_excel(integrated_path)
    except ValueError:
        # 혹시 확장자는 xlsx인데 실제 내용이 csv일 경우를 대비한 폴백 처리
        df_integrated = pd.read_csv(integrated_path)
        
    df_nas = pd.read_csv(nas_db_path)
    
    # 결측치(NaN) 방지 (문자열 처리를 위해)
    df_integrated['OCR결과'] = df_integrated['OCR결과'].fillna('')
    df_nas['파일명'] = df_nas['파일명'].fillna('')
    
    # 매칭 결과를 담을 리스트
    matched_results = []
    
    print("NAS 파일 매칭을 시작합니다...")
    # 2. integrated의 각 'OCR결과' 값에 대해 검색 수행
    for ocr_text in tqdm(df_integrated['OCR결과'], desc="매칭 진행률"):
        ocr_text = str(ocr_text).strip()
        
        if not ocr_text:  # 빈 칸인 경우 건너뜀
            matched_results.append("")
            continue
            
        # NAS 파일명에 OCR결과 텍스트가 포함(contains)되어 있는지 확인 (대소문자 구분 없음)
        matches = df_nas[df_nas['파일명'].str.contains(ocr_text, case=False, regex=False, na=False)]
        
        if not matches.empty:
            # [수정된 부분] 전체경로 대신 '파일명'들만 가져와서 줄바꿈(\n)으로 묶어 저장
            matched_files = "\n".join(matches['파일명'].tolist())
            matched_results.append(matched_files)
        else:
            matched_results.append("")  # 일치하는 항목이 없을 경우 빈 칸 유지
            
    # 3. 매칭된 결과 리스트를 원본 데이터프레임의 'NAS 매칭' 칼럼에 덮어쓰기
    df_integrated['NAS 매칭'] = matched_results
    
    # 4. 결과를 새로운 엑셀 파일로 저장
    print(f"\n매칭 완료! 결과를 '{output_path}'에 저장하는 중...")
    df_integrated.to_excel(output_path, index=False)
    print("모든 작업이 성공적으로 끝났습니다!")

if __name__ == "__main__":
    # 파일 경로 설정 (실제 환경에 맞게 경로를 수정해 주세요)
    INTEGRATED_FILE = r'output\integrated.xlsx'           # 원본 파일 (CSV인 경우 경로 수정 필요)
    NAS_DB_FILE = 'NAS_File_DB.csv'                       # 1단계에서 만든 NAS DB 파일
    OUTPUT_FILE = r'output\nas_matched.xlsx'       # 결과를 저장할 새로운 엑셀 파일 경로
    
    match_ocr_to_nas(INTEGRATED_FILE, NAS_DB_FILE, OUTPUT_FILE)

데이터를 불러오는 중...
NAS 파일 매칭을 시작합니다...


매칭 진행률: 100%|██████████| 918/918 [00:34<00:00, 26.43it/s]



매칭 완료! 결과를 'output\integrated_matched.xlsx'에 저장하는 중...
모든 작업이 성공적으로 끝났습니다!


국립중앙도서관 검색결과

In [ ]:
import openpyxl
import requests
import re
import toml
from difflib import SequenceMatcher
import time

# 1. 텍스트 정규화 함수 (검색용 텍스트 클리닝)
def clean_text_for_search(text):
    if not isinstance(text, str):
        return ""
    # 괄호 및 괄호 안의 부가 내용 제거 (예: (상), [III], <I>)
    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>|【.*?】', '', text)
    # 로마자 등 불필요한 특수문자 제거 및 공백으로 치환
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9]', ' ', text)
    # 연속된 공백을 하나로 통일하고 양끝 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. 유사도 계산 함수 (띄어쓰기 무시)
def get_similarity(a, b):
    if not a or not b: return 0
    a_norm = a.replace(" ", "")
    b_norm = b.replace(" ", "")
    return SequenceMatcher(None, a_norm, b_norm).ratio()

# 3. 메인 실행 함수
def update_excel_with_nl_api():
    # secrets.toml에서 API 키 로드
    try:
        config = toml.load('config/secrets.toml')
        api_key = config.get('NL_KEY')
        if not api_key:
            raise ValueError("NL_KEY가 존재하지 않습니다.")
    except Exception as e:
        print(f"API 키 로드 실패: {e}")
        return

    # 파일 로드 (openpyxl을 사용하여 기존 칼럼과 수식 유지)
    file_path = 'output/nas_matched.xlsx'
    try:
        wb = openpyxl.load_workbook(file_path)
        ws = wb.active
    except Exception as e:
        print(f"엑셀 파일 로드 실패: {e}")
        return

    # 첫 번째 행(헤더)의 칼럼 위치 파악
    header = {cell.value: cell.column for cell in ws[1]}
    col_ocr = header.get('OCR결과')
    col_nl_match = header.get('NL 매칭')
    col_author = header.get('저작자')
    col_publisher = header.get('발행자')

    # 국립중앙도서관 일반검색 요청 URL [cite: 27]
    api_url = "https://www.nl.go.kr/NL/search/openApi/search.do"

    print("API 검색 및 데이터 업데이트 시작...")
    
    # 2행부터 마지막 행까지 순회
    for row in range(2, ws.max_row + 1):
        ocr_raw = ws.cell(row=row, column=col_ocr).value
        
        # 값이 비어있거나 이미 NL 매칭 결과가 채워져 있다면 스킵
        if not ocr_raw or ws.cell(row=row, column=col_nl_match).value:
            continue
            
        search_kwd = clean_text_for_search(ocr_raw)
        if len(search_kwd) < 2:
            continue

        # 가이드라인에 맞춘 API 호출 파라미터 [cite: 34]
        params = {
            'key': api_key,         # 발급키 [cite: 34]
            'kwd': search_kwd,      # 검색어 [cite: 34]
            'pageNum': 1,           # 현재페이지 (필수) [cite: 34]
            'pageSize': 5,          # 쪽당출력건수 (유사도 비교를 위해 상위 5개 호출) [cite: 34]
            'apiType': 'json',      # 반환 형식 (xml, json 중 json 선택) [cite: 36]
            'srchTarget': 'title'   # 제목 검색으로 범위 한정 [cite: 34]
        }

        try:
            response = requests.get(api_url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                results = data.get('result', [])
                
                best_match = None
                highest_sim = 0
                
                # 반환된 상위 5개 결과 중 OCR 원본과 가장 유사도가 높은 책 찾기
                for item in results:
                    api_title = item.get('title_info', '') # 표제 리스트 
                    # API 반환값에 <b>태그나 기타 특수문자가 섞여 있을 수 있으므로 클리닝
                    clean_api_title = re.sub(r'<.*?>', '', api_title) 
                    
                    sim = get_similarity(search_kwd, clean_api_title)
                    if sim > highest_sim:
                        highest_sim = sim
                        best_match = item
                        
                # 유사도가 0.65 (65%) 이상인 경우에만 결과 채택
                if best_match and highest_sim > 0.65:
                    clean_best_title = re.sub(r'<.*?>', '', best_match.get('title_info', '')) # 표제 
                    
                    # 엑셀 셀 업데이트 (다른 칼럼은 건드리지 않음)
                    ws.cell(row=row, column=col_nl_match).value = clean_best_title
                    ws.cell(row=row, column=col_author).value = best_match.get('author_info', '') # 저작자 
                    ws.cell(row=row, column=col_publisher).value = best_match.get('pub_info', '') # 발행자 
                    
                    print(f"[행 {row}] 매칭성공 (유사도 {highest_sim:.2f}): {clean_best_title}")
                else:
                    print(f"[행 {row}] 일치 항목 없음 (검색어: {search_kwd})")

            # 국립중앙도서관 API 서버 부하를 방지하기 위해 0.2초 대기
            time.sleep(0.2)
            
        except Exception as e:
            print(f"[행 {row}] API 요청 중 에러 발생: {e}")

    # 기존 포맷과 매크로(함수)를 그대로 유지한 채 덮어쓰기 저장
    wb.save(file_path)
    print("작업 완료! 파일이 성공적으로 저장되었습니다.")

if __name__ == "__main__":
    update_excel_with_nl_api()

NL API 테스트

In [5]:
# 파일명: test_nl_api.py
# 지시사항: 9개의 샘플 책 제목을 국립중앙도서관 API(HTTP 우회 모드)로 검색하여 결과를 출력합니다.

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import urllib3
import re
import toml
from difflib import SequenceMatcher
import time

# SSL 경고 메시지 숨기기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def clean_text_for_search(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\[.*?\]|\(.*?\)|<.*?>|【.*?】', '', text)
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_similarity(a, b):
    if not a or not b: return 0
    a_norm = a.replace(" ", "")
    b_norm = b.replace(" ", "")
    return SequenceMatcher(None, a_norm, b_norm).ratio()

def test_api():
    # API 키 불러오기
    try:
        config = toml.load('config/secrets.toml')
        api_key = config.get('NL_KEY')
        if not api_key:
            print("오류: NL_KEY가 존재하지 않습니다.")
            return
    except Exception as e:
        print(f"API 키 로드 실패: {e}")
        return

    # 테스트할 9개(중복 포함)의 책 제목 리스트
    test_queries = [
        "시흥시의 출범과 성장",
        "道德經",
        "Quality of Life",
        "한의원단위 증례연구 활성화를 위한 증례보고정리집",
        "나는 침과 뜸으로 승부한다",
        "활인심방",
        "활인심방",
        "인문학적 마음유지와 한국의학의 만남",
        "도서 문화유적지 지표조사 및 자원화연구 신의면편"
    ]

    # HTTPS 대신 HTTP 사용
    api_url = "http://www.nl.go.kr/NL/search/openApi/search.do"

    # 세션 및 재시도 설정 (안정성 확보)
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    print("=== 국립중앙도서관 API 검색 테스트 시작 (HTTP 우회 모드) ===\n")

    for original_title in test_queries:
        search_kwd = clean_text_for_search(original_title)
        
        print(f"▶ 원본: {original_title}")
        print(f"  검색어: {search_kwd}")

        params = {
            'key': api_key,
            'kwd': search_kwd,
            'pageNum': 1,
            'pageSize': 5,
            'apiType': 'json',
            'srchTarget': 'title'
        }

        try:
            # verify=False 추가로 SSL 인증서 문제 회피
            response = session.get(api_url, params=params, headers=headers, timeout=10, verify=False)
            
            if response.status_code == 200:
                data = response.json()
                
                # 검색 결과가 0건인 경우 처리
                if data.get('total', 0) == 0:
                    print(f"  [결과없음] 도서관 DB에 검색어와 일치하는 책이 없습니다.")
                    print("-" * 50)
                    continue

                results = data.get('result', [])
                
                best_match = None
                highest_sim = 0
                
                for item in results:
                    api_title = item.get('title_info', '')
                    clean_api_title = re.sub(r'<.*?>', '', api_title) 
                    
                    sim = get_similarity(search_kwd, clean_api_title)
                    if sim > highest_sim:
                        highest_sim = sim
                        best_match = item
                        
                if best_match and highest_sim > 0.65:
                    clean_best_title = re.sub(r'<.*?>', '', best_match.get('title_info', ''))
                    author = best_match.get('author_info', '정보없음')
                    publisher = best_match.get('pub_info', '정보없음')
                    
                    print(f"  [성공] 매칭 결과: {clean_best_title}")
                    print(f"         저작자: {author}")
                    print(f"         발행처: {publisher}")
                    print(f"         유사도: {highest_sim:.2f}")
                else:
                    print(f"  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.")
            else:
                print(f"  [에러] API 상태 코드: {response.status_code}")

        except Exception as e:
            print(f"  [에러] 네트워크 예외 발생: {e}")
            
        print("-" * 50)
        time.sleep(0.5) # 서버 부하 방지 대기

if __name__ == "__main__":
    test_api()

=== 국립중앙도서관 API 검색 테스트 시작 (HTTP 우회 모드) ===

▶ 원본: 시흥시의 출범과 성장
  검색어: 시흥시의 출범과 성장
  [결과없음] 도서관 DB에 검색어와 일치하는 책이 없습니다.
--------------------------------------------------
▶ 원본: 道德經
  검색어: 道德經
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: Quality of Life
  검색어: Quality of Life
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: 한의원단위 증례연구 활성화를 위한 증례보고정리집
  검색어: 한의원단위 증례연구 활성화를 위한 증례보고정리집
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: 나는 침과 뜸으로 승부한다
  검색어: 나는 침과 뜸으로 승부한다
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: 활인심방
  검색어: 활인심방
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: 활인심방
  검색어: 활인심방
  [실패] 유사도 65% 이상인 적절한 매칭 결과를 찾지 못했습니다.
--------------------------------------------------
▶ 원본: 인문학적 마음유지와 한국의학의 만남
  검색어: 인문학적 마음유지와 한국의학의 만남
  [결과없음] 도서

ㅁㅁ

In [3]:
import requests
import re
import toml
import urllib3
from difflib import SequenceMatcher

# SSL 경고 메시지 숨기기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def test_api_connection():
    try:
        config = toml.load('config/secrets.toml')
        api_key = config.get('NL_KEY')
    except Exception as e:
        print(f"API 키 로드 실패: {e}")
        return

    # 테스트 검색어
    search_kwd = "시흥시의 출범과 성장"
    
    # 1. https 대신 http로 변경해보기 (공공기관 서버 테스트용)
    api_url = "http://www.nl.go.kr/NL/search/openApi/search.do"
    
    params = {
        'key': api_key,
        'kwd': search_kwd,
        'pageNum': 1,
        'pageSize': 5,
        'apiType': 'json',
        'srchTarget': 'title'
    }

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    print(f"[{search_kwd}] 검색 연결 테스트 중...\n")

    try:
        # verify=False 를 추가하여 SSL 인증서 검증을 건너뜀
        response = requests.get(api_url, params=params, headers=headers, timeout=10, verify=False)
        
        print(f"서버 응답 코드: {response.status_code}")
        if response.status_code == 200:
            print("연결 성공! API가 정상적으로 데이터를 반환했습니다.")
            print("응답 데이터 일부:", response.text[:200])
        else:
            print("연결은 되었으나 에러가 반환되었습니다.")
            print(response.text)
            
    except requests.exceptions.ConnectTimeout:
        print("❌ [오류] ConnectTimeout 발생.")
        print("이것은 파라미터 문제가 아니라, 현재 사용 중인 PC의 네트워크(방화벽/VPN)가 국립중앙도서관 접속을 차단하고 있거나, 도서관 서버가 현재 PC의 IP를 막고 있다는 뜻입니다.")
        print("해결책: 일반 가정용 와이파이나 스마트폰 테더링으로 네트워크를 변경한 후 다시 실행해 보세요.")
    except Exception as e:
        print(f"❌ 기타 네트워크 오류: {e}")

if __name__ == "__main__":
    test_api_connection()

[시흥시의 출범과 성장] 검색 연결 테스트 중...

서버 응답 코드: 200
연결 성공! API가 정상적으로 데이터를 반환했습니다.
응답 데이터 일부: {"total":0,"kwd":"시흥시의 출범과 성장","pageNum":1,"pageSize":5,"category":"","sort":""}


In [2]:
# 파일명: convert_nas_path.py
# 지시사항: NAS_File.csv 파일을 불러와 '전체경로'의 루트 폴더(2015이후백업, AI_MCP)를 실제 네트워크 드라이브(Z:\, Y:\)로 변경하여 '실제 경로' 칼럼을 추가하고, 새로운 엑셀 파일로 저장합니다.

import pandas as pd

# 실제 경로로 변환하는 함수
def make_real_path(path):
    if not isinstance(path, str):
        return path
    
    # '2015이후백업'으로 시작하면 Z: 로 교체
    # 예: 2015이후백업\문서.hwp -> Z:\문서.hwp
    if path.startswith('2015이후백업'):
        return path.replace('2015이후백업', 'Z:', 1)
    
    # 'AI_MCP'로 시작하면 Y: 로 교체
    elif path.startswith('AI_MCP'):
        return path.replace('AI_MCP', 'Y:', 1)
    
    # 조건에 해당하지 않으면 원래 경로 반환
    return path

def main():
    input_file = 'NAS_File_DB.csv'
    output_file = 'NAS_File_with_realpath.xlsx'

    print("데이터를 불러오는 중...")
    # 한글 깨짐을 방지하기 위해 여러 인코딩 시도
    try:
        df = pd.read_csv(input_file, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df = pd.read_csv(input_file, encoding='cp949')

    print("'실제 경로' 칼럼을 생성하는 중...")
    # '전체경로' 칼럼에 변환 함수를 적용하여 새로운 칼럼 생성
    df['실제 경로'] = df['전체경로'].apply(make_real_path)

    print("새로운 엑셀 파일로 저장하는 중...")
    # 엑셀 파일로 저장 (인덱스 번호 제외)
    df.to_excel(output_file, index=False)

    print(f"🎉 작업 완료! '{output_file}' 파일이 성공적으로 생성되었습니다.")

if __name__ == "__main__":
    main()

데이터를 불러오는 중...
'실제 경로' 칼럼을 생성하는 중...
새로운 엑셀 파일로 저장하는 중...
🎉 작업 완료! 'NAS_File_with_realpath.xlsx' 파일이 성공적으로 생성되었습니다.
